# Дослідний аналіз коротких напливів
Джерело: схвалений власником локальний експорт за 04.09–01.10.2026. КПП 1, 6, 17, 29. Всі години Europe/Kyiv. Тільки чисте зростання активної черги, не кількість реєстрацій. Модель production та якість голосових попереджень не підтверджуються.

In [ ]:
"""Exploratory chronological evaluation; local approved operational extract only.

No production code, no network calls, no forecasting accuracy claim.
Daily maxima are measured net queue growth, not registration events.
"""
import bisect, gzip, hashlib, json
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean, median
from zoneinfo import ZoneInfo

ROOT=Path('C:/AI/AvelRen/dist')
source=ROOT/'queue-surge-28days.json.gz'
rows=json.loads(gzip.decompress(source.read_bytes()))
zone=ZoneInfo('Europe/Kyiv')
groups=defaultdict(list)
for cp,t,count,paused in rows:
    at=datetime.fromtimestamp(t,timezone.utc).astimezone(zone)
    groups[(cp,at.date().isoformat())].append((t,count,paused,at))
report={'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'records':len(rows),'training_rule':'only complete dates preceding forecast date; issue at 00:00 Kyiv',
        'evaluation_dates':'2026-09-18 through 2026-10-01',
        'exploratory_not_untouched_holdout':True,
        'quality_rule':'at least 95% of nominal minute observations and 90% valid intraday windows',
        'day_peaks':[],'metrics':{},'chop_oct1_predictions':{}}
for (cp,day),samples in sorted(groups.items()):
    samples.sort()
    times=[s[0] for s in samples]
    broken=[0]
    for i in range(1,len(times)):
        broken.append(broken[-1]+int(times[i]-times[i-1]>90))
    for window in (5,10,20,60):
        candidates=[]
        for j,s in enumerate(samples):
            target=s[0]-window*60
            k=bisect.bisect_left(times,target)
            choices=[i for i in (k-1,k) if 0<=i<j]
            if not choices: continue
            i=min(choices,key=lambda i:abs(times[i]-target))
            if abs(times[i]-target)>30 or broken[j]!=broken[i]: continue
            candidates.append((s[1]-samples[i][1],i,j))
        if not candidates: continue
        delta,i,j=max(candidates,key=lambda x:(x[0],-x[1]))
        start=samples[i][3]
        report['day_peaks'].append({'checkpoint_id':cp,'date':day,'window':window,
            'samples':len(samples),'valid_windows':len(candidates),
            'eligible':len(samples)>=1440*.95 and len(candidates)>=(1440-window)*.90,
            'increase':delta,'start':start.isoformat(),'end':samples[j][3].isoformat(),
            'minute':start.hour*60+start.minute,'weekday':start.weekday(),
            'paused_at_start':samples[i][2],
            'pause_transition':any(samples[k][2]!=samples[k-1][2] for k in range(i+1,j+1))})

def typical_minute(events):
    # Circular medoid prevents e.g. 23:55 and 00:05 averaging to noon.
    values=[e['minute'] for e in events]
    return min(sorted(set(values)),key=lambda x:sum(min(abs(x-y),1440-abs(x-y)) for y in values))

for cp in (17,1,6,29):
    report['metrics'][str(cp)]={}
    for window in (5,10,20,60):
        all_events=[e for e in report['day_peaks'] if e['checkpoint_id']==cp and e['window']==window]
        scores=defaultdict(list)
        for event in all_events:
            if event['date']<'2026-09-18' or not event['eligible']: continue
            prior=[e for e in all_events if e['date']<event['date'] and e['eligible']]
            if len(prior)<7: continue
            methods={'previous_eligible_day':prior[-1:], 'typical_last14_days':prior[-14:]}
            weekdays=[e for e in prior if e['weekday']==event['weekday']]
            if len(weekdays)>=2: methods['same_weekday']=weekdays
            for name,train in methods.items():
                predicted_minute=typical_minute(train)
                predicted_delta=median(e['increase'] for e in train)
                error=abs(predicted_minute-event['minute'])
                scores[name].append({'date':event['date'],'time_error_minutes':error,
                    'growth_error':abs(predicted_delta-event['increase'])})
                if cp==17 and event['date']=='2026-10-01':
                    report['chop_oct1_predictions'].setdefault(str(window),{})[name]={
                        'issued_at':'2026-10-01T00:00:00+03:00',
                        'predicted_start':f'{predicted_minute//60:02d}:{predicted_minute%60:02d}',
                        'predicted_growth':predicted_delta,'time_error_minutes':error,
                        'actual_growth':event['increase'],'actual_start':event['start']}
        report['metrics'][str(cp)][str(window)]={name:{
            'cases':len(values),
            'time_mae_minutes':round(mean(v['time_error_minutes'] for v in values),1),
            'growth_mae':round(mean(v['growth_error'] for v in values),1),
            'within_15_minutes':sum(v['time_error_minutes']<=15 for v in values),
            'within_60_minutes':sum(v['time_error_minutes']<=60 for v in values)}
            for name,values in scores.items()}
out=ROOT/'queue-surge-comparison.json'
out.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in report.items() if k!='day_peaks'},ensure_ascii=True,indent=2))
print('CHOP DAILY 5M PEAKS')
for e in report['day_peaks']:
    if e['checkpoint_id']==17 and e['window']==5:
        print(e['date'],e['start'][11:16],e['increase'],e['eligible'])


Обмеження: переглянута дослідна вибірка, не фінальний holdout; максимуми спокійних днів також входять до оцінки. Пороги події і метрики precision/recall ще не погоджені. Повторно запускати лише на вже дозволеному локальному артефакті; цей notebook не звертається до сервера.

# Дослідне порівняння трьох моделей
Тимчасова подія: +20 авто за 5 хв. Поріг визначається на 18–24.09, дослідне порівняння на 25.09–01.10. Період уже переглянутий; це не фінальний holdout. Потрібна наявна NumPy. Жодних мережевих запитів чи product-коду.

In [ ]:
import sys
sys.argv = ['surge-model-probe.py', '--older', '--advanced']
"""Throwaway offline probe, not application code or a validated forecast.

Event: >=20 net vehicles within 5 minutes. Overlapping/gap<=30 minute
windows form one event. Compare calendar, momentum, and regularized logistic
scores. Train/validation/test are chronological with purged label horizons.
No requests to the source, server writes, or probability accuracy claim.
"""
import argparse, bisect, gzip, hashlib, json, math
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from zoneinfo import ZoneInfo

ROOT=Path('C:/AI/AvelRen/dist')
source=ROOT/'queue-surge-28days.json.gz'
raw=json.loads(gzip.decompress(source.read_bytes()))
parser=argparse.ArgumentParser()
parser.add_argument('--older',action='store_true')
parser.add_argument('--advanced',action='store_true')
args=parser.parse_args()
source_hashes={source.name:hashlib.sha256(source.read_bytes()).hexdigest()}
if args.older:
    older=ROOT/'queue-surge-older-20260806-0903.json.gz'
    raw=json.loads(gzip.decompress(older.read_bytes()))+raw
    source_hashes[older.name]=hashlib.sha256(older.read_bytes()).hexdigest()
assert len(raw)==len({(r[0],r[1]) for r in raw}), 'overlapping source rows'
zone=ZoneInfo('Europe/Kyiv')
def stamp(s): return int(datetime.fromisoformat(s).timestamp())
START=stamp('2026-09-04T00:00:00+03:00')
VALIDATION=stamp('2026-09-18T00:00:00+03:00')
TEST=stamp('2026-09-25T00:00:00+03:00')
END=stamp('2026-10-02T00:00:00+03:00')
data=defaultdict(list)
for cp,t,c,p in raw: data[cp].append((t,c,p))
CP=(1,6,17,29)
report={'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'source_hashes':source_hashes,'source_records':len(raw),
        'definition':'net increase >=20 in 5 minutes; windows within 30 minutes clustered',
        'splits_kyiv':['train Aug06-Sep17' if args.older else 'train Sep04-17','validation Sep18-24','test Sep25-Oct01'],
        'limitation':'already inspected exploratory period, not untouched final holdout; no calibrated probabilities',
        'future_quality':'>=95% expected minute samples; no gap >5min; endpoint tolerance 30 seconds',
        'results':{},'events':{}}
series={}
for cp,rows in data.items():
    rows.sort()
    t=np.array([r[0] for r in rows],dtype=np.int64)
    c=np.array([r[1] for r in rows],dtype=float)
    p=np.array([r[2] for r in rows],dtype=float)
    bad=np.concatenate(([0],np.cumsum(np.diff(t)>90)))
    severe=np.concatenate(([0],np.cumsum(np.diff(t)>300)))
    starts=[]
    for j,end in enumerate(t):
        i=bisect.bisect_left(t,end-300)
        choices=[k for k in (i-1,i) if 0<=k<j]
        if not choices: continue
        i=min(choices,key=lambda k:abs(int(t[k])-int(end-300)))
        if abs(t[i]-(end-300))<=30 and bad[j]==bad[i] and c[j]-c[i]>=20:
            start=int(t[i])
            if not starts or start>last_window+1800: starts.append(start)
            last_window=start
    series[cp]=(t,c,p,bad,severe,np.array(starts,dtype=np.int64))
    report['events'][str(cp)]=[datetime.fromtimestamp(x,timezone.utc).astimezone(zone).isoformat() for x in starts]

def samples(horizon_min,lead_min):
    output=[]
    for cp,(t,c,p,bad,severe,events) in series.items():
        for j,origin in enumerate(t):
            local=datetime.fromtimestamp(int(origin),timezone.utc).astimezone(zone)
            if horizon_min==1440:
                if local.hour!=0 or local.minute!=0: continue
            elif local.minute%5: continue
            boundary=VALIDATION if origin<VALIDATION else TEST if origin<TEST else END
            future_end=int(origin)+horizon_min*60
            # Extra 5min are needed to verify the event window at horizon end.
            label_end=future_end+300
            if label_end>=boundary: continue
            lag=[]
            for minute in (5,10,20,60):
                goal=int(origin)-minute*60
                i=bisect.bisect_left(t,goal)
                options=[k for k in (i-1,i) if 0<=k<=j]
                if not options: break
                k=min(options,key=lambda k:abs(int(t[k])-goal))
                if abs(t[k]-goal)>30 or bad[k]!=bad[j]: break
                lag.append(float(c[j]-c[k]))
            if len(lag)!=4: continue
            k=bisect.bisect_left(t,label_end)
            if k>=len(t) or abs(t[k]-label_end)>30: continue
            expected=(label_end-int(origin))/60+1
            if (k-j+1)/expected<.95 or severe[k]!=severe[j]: continue
            phase=local.hour*60+local.minute
            features=[c[j],*lag,lag[0]-lag[1]/2,p[j],
                      math.sin(2*math.pi*phase/1440),math.cos(2*math.pi*phase/1440),
                      math.sin(2*math.pi*local.weekday()/7),math.cos(2*math.pi*local.weekday()/7),
                      *(int(cp==x) for x in CP)]
            eligible=events[(events>=int(origin)+lead_min*60)&(events<=future_end)]
            output.append({'cp':cp,'origin':int(origin),'features':features,'y':int(len(eligible)>0),
                           'events':eligible.tolist(),'weekday':local.weekday(),'hourbin':local.hour//3})
    return output

def score_alerts(rows,scores,threshold,lead,horizon):
    alerts=[]; last={}
    for row,score in sorted(zip(rows,scores),key=lambda x:x[0]['origin']):
        if score<threshold: continue
        cp,origin=row['cp'],row['origin']
        if origin-last.get(cp,-10**20)<3600: continue
        alerts.append(row); last[cp]=origin
    eligible={(r['cp'],e) for r in rows for e in r['events']}
    matched=set(); successes=[]; matched_details=[]
    for r in alerts:
        possibilities=sorted(e for cp,e in eligible if cp==r['cp'] and
                    r['origin']+lead*60<=e<=r['origin']+horizon*60 and (cp,e) not in matched)
        if possibilities:
            e=possibilities[0]; matched.add((r['cp'],e)); successes.append((e-r['origin'])/60)
            matched_details.append({'checkpoint_id':r['cp'],
                'issued_at':datetime.fromtimestamp(r['origin'],timezone.utc).astimezone(zone).isoformat(),
                'event_start':datetime.fromtimestamp(e,timezone.utc).astimezone(zone).isoformat(),
                'lead_minutes':(e-r['origin'])/60})
    precision=len(matched)/len(alerts) if alerts else 0
    recall=len(matched)/len(eligible) if eligible else 0
    return {'alerts':len(alerts),'correct_alerts':len(matched),'false_alerts':len(alerts)-len(matched),
            'eligible_events':len(eligible),'detected_events':len(matched),
            'precision':round(precision,4),'recall':round(recall,4),
            'f1':2*precision*recall/(precision+recall) if precision+recall else 0,
            'mean_lead_minutes':round(float(np.mean(successes)),1) if successes else None,
            'detected_chop_oct1_afternoon':any(cp==17 and datetime.fromtimestamp(e,timezone.utc).astimezone(zone).date().isoformat()=='2026-10-01'
                and 17<=datetime.fromtimestamp(e,timezone.utc).astimezone(zone).hour<18 for cp,e in matched),
            'matched_events':matched_details}

for horizon,lead in ((60,15),(1440,60)):
    rows=samples(horizon,lead)
    train=[r for r in rows if r['origin']<VALIDATION]
    val=[r for r in rows if VALIDATION<=r['origin']<TEST]
    test=[r for r in rows if r['origin']>=TEST]
    print('DATA',horizon,[(len(x),sum(r['y'] for r in x)) for x in (train,val,test)],flush=True)
    if not train or not val or not test or not sum(r['y'] for r in train):
        report['results'][str(horizon)]={'status':'insufficient_labels'};continue
    x=np.array([r['features'] for r in train]); y=np.array([r['y'] for r in train],dtype=float)
    mu=x.mean(axis=0); sd=x.std(axis=0);sd[sd<1e-8]=1
    def matrix(rs): return np.column_stack((np.ones(len(rs)),(np.array([r['features'] for r in rs])-mu)/sd))
    design=matrix(train); w=np.zeros(design.shape[1]);
    weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
    for step in range(500):
        probability=1/(1+np.exp(-np.clip(design@w,-30,30)))
        gradient=design.T@((probability-y)*weights)/len(y)
        gradient[1:]+=.03*w[1:]
        w-=.08*gradient
    bins=defaultdict(list)
    for r in train:bins[(r['cp'],r['weekday'],r['hourbin'])].append(r['y'])
    base=float(y.mean())
    def calendar(rs):
        return np.array([(sum(bins[(r['cp'],r['weekday'],r['hourbin'])])+5*base)/(len(bins[(r['cp'],r['weekday'],r['hourbin'])])+5) for r in rs])
    def momentum(rs):return np.array([max(0,r['features'][1]/20,r['features'][2]/40) for r in rs])
    metrics={'split_samples':[len(train),len(val),len(test)],'positive_samples':[int(y.sum()),sum(r['y'] for r in val),sum(r['y'] for r in test)],'methods':{}}
    methods=[('calendar',calendar),('momentum',momentum),('logistic',lambda rs:1/(1+np.exp(-np.clip(matrix(rs)@w,-30,30))))]
    if args.advanced and horizon==60:
        from scipy.optimize import minimize
        def expanded(rs):
            z=(np.array([r['features'] for r in rs])-mu)/sd
            z=np.clip(z,-8,8)
            # CP-specific behavior plus curvature; transforms are fixed before test.
            return np.column_stack((np.ones(len(rs)),z,z[:,:7]**2,
                *(z[:,:7]*z[:,k:k+1] for k in range(11,15))))
        qx=expanded(train)
        def objective(coef):
            logits=qx@coef
            loss=float(np.mean(weights*(np.logaddexp(0,logits)-y*logits))+.03/2*np.sum(coef[1:]**2))
            prob=1/(1+np.exp(-np.clip(logits,-30,30)))
            grad=qx.T@((prob-y)*weights)/len(y)
            grad[1:]+=.03*coef[1:]
            return loss,grad
        fit=minimize(objective,np.zeros(qx.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':200,'ftol':1e-9})
        metrics['nonlinear_fit']={'converged':bool(fit.success),'iterations':int(fit.nit),'features':int(qx.shape[1])}
        methods.append(('nonlinear_logistic',lambda rs:1/(1+np.exp(-np.clip(expanded(rs)@fit.x,-30,30)))))

        # Histogram Newton boosting with decision stumps: research baseline only.
        cuts=[np.unique(np.quantile(x[:,k],np.linspace(.05,.95,19))) for k in range(x.shape[1])]
        indices=[np.searchsorted(cuts[k],x[:,k],side='left') for k in range(x.shape[1])]
        logits=np.zeros(len(y)); stumps=[]
        for iteration in range(80):
            prob=1/(1+np.exp(-np.clip(logits,-30,30)))
            gradient=weights*(y-prob); hessian=weights*prob*(1-prob)
            best=None
            for k,histogram_bins in enumerate(indices):
                size=len(cuts[k])+1
                gs=np.bincount(histogram_bins,weights=gradient,minlength=size).cumsum()[:-1]
                hs=np.bincount(histogram_bins,weights=hessian,minlength=size).cumsum()[:-1]
                ns=np.bincount(histogram_bins,minlength=size).cumsum()[:-1]
                rg=gradient.sum()-gs;rh=hessian.sum()-hs
                gain=gs**2/(hs+20)+rg**2/(rh+20)
                gain[(ns<120)|(len(y)-ns<120)]=-np.inf
                split=int(np.argmax(gain))
                if not np.isfinite(gain[split]):continue
                candidate=(float(gain[split]),k,split,float(gs[split]/(hs[split]+20)),float(rg[split]/(rh[split]+20)))
                if best is None or candidate[0]>best[0]:best=candidate
            if best is None:break
            _,k,split,left,right=best
            cutoff=float(cuts[k][split]);left*=.1;right*=.1
            stumps.append((k,cutoff,left,right))
            logits+=np.where(x[:,k]<=cutoff,left,right)
        def boosted(rs):
            values=np.array([r['features'] for r in rs]);scores=np.zeros(len(rs))
            for k,cutoff,left,right in stumps:scores+=np.where(values[:,k]<=cutoff,left,right)
            return 1/(1+np.exp(-np.clip(scores,-30,30)))
        methods.append(('boosted_stumps',boosted))
        metrics['boosting_fit']={'trees':len(stumps),'leaf_minimum':120,'learning_rate':.1}
    for name,fn in methods:
        vs=fn(val);ts=fn(test)
        thresholds=sorted(set([0.0,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,41)).tolist()]))
        choices=[(score_alerts(val,vs,threshold,lead,horizon),threshold) for threshold in thresholds]
        selected,threshold=max(choices,key=lambda x:(x[0]['f1'],x[0]['precision'],-x[0]['alerts']))
        test_result=score_alerts(test,ts,threshold,lead,horizon)
        metrics['methods'][name]={'threshold':round(threshold,5),'validation':selected,'test':test_result}
        print('RESULT',horizon,name,json.dumps(test_result),flush=True)
        conservative=[(result,cutoff) for result,cutoff in choices
                      if result['precision']>=.99 and result['correct_alerts']>0]
        if conservative:
            cv,ct=max(conservative,key=lambda x:(x[0]['recall'],-x[0]['alerts']))
            cr=score_alerts(test,ts,ct,lead,horizon)
            metrics['methods'][name]['validation_99_policy']={
                'threshold':round(ct,5),'validation':cv,'test':cr,
                'limitation':'selected on few validation events; not a 99% guarantee'}
            print('CONSERVATIVE',horizon,name,json.dumps(cr),flush=True)
        else:
            metrics['methods'][name]['validation_99_policy']={'status':'no threshold with nonzero validated alerts and >=99% precision'}
            print('CONSERVATIVE',horizon,name,'unavailable',flush=True)
    metrics['selected_by_validation']=max(metrics['methods'],key=lambda name:(metrics['methods'][name]['validation']['f1'],metrics['methods'][name]['validation']['precision'],-metrics['methods'][name]['validation']['alerts']))
    report['results'][str(horizon)]=metrics
result_path=ROOT/('surge-model-probe-results-advanced.json' if args.advanced else 'surge-model-probe-results-extended.json' if args.older else 'surge-model-probe-results.json')
result_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('SAVED',result_path,flush=True)


# Описова перевірка передвісників
Фонові вікна залежні та не узгоджені за часом доби. Це не причинність і не precision/recall прогнозної моделі.

In [ ]:
"""Descriptive precursor audit on approved local data; not an alert rule."""
import bisect,gzip,json
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from statistics import median

root=Path('C:/AI/AvelRen/dist')
data=defaultdict(list)
for filename in ('queue-surge-older-20260806-0903.json.gz','queue-surge-28days.json.gz'):
    for cp,t,c,p in json.loads(gzip.decompress((root/filename).read_bytes())):
        data[cp].append((t,c,p))
events=json.loads((root/'surge-model-probe-results-advanced.json').read_text())['events']
details=[];controls={}
for cp,labels in events.items():
    rows=sorted(data[int(cp)]);times=[r[0] for r in rows]
    def point(target):
        i=bisect.bisect_left(times,target)
        choices=[j for j in (i-1,i) if 0<=j<len(times)]
        if not choices:return None
        j=min(choices,key=lambda j:abs(times[j]-target))
        return j if abs(times[j]-target)<=30 else None
    event_times=sorted(int(datetime.fromisoformat(label).timestamp()) for label in labels)
    if event_times:
        background=[]
        for end,row in enumerate(rows):
            t=row[0]
            if (t//60)%5:continue
            k=bisect.bisect_left(event_times,t)
            if any(abs(event_times[j]-t)<=75*60 for j in (k-1,k) if 0<=j<len(event_times)):continue
            begin=point(t-300)
            if begin is None or any(times[j]-times[j-1]>90 for j in range(begin+1,end+1)):continue
            background.append({'change_last5':row[1]-rows[begin][1],
                'pause_changed_last5':any(rows[j][2]!=rows[j-1][2] for j in range(begin+1,end+1))})
        controls[cp]={'windows':len(background),'median_last5_growth':median(v['change_last5'] for v in background),
            'growing_5plus':sum(v['change_last5']>=5 for v in background),
            'pause_changed_last5':sum(v['pause_changed_last5'] for v in background)}
    for label in labels:
        start=int(datetime.fromisoformat(label).timestamp());i=point(start)
        if i is None:continue
        result={'checkpoint_id':int(cp),'event_start':label,'before':{}}
        for lead in (15,30,60):
            end=point(start-lead*60);begin=point(start-(lead+5)*60)
            if end is None or begin is None or any(times[k]-times[k-1]>90 for k in range(begin+1,end+1)):
                result['before'][str(lead)]=None;continue
            result['before'][str(lead)]={'change_last5':rows[end][1]-rows[begin][1],
                'paused':rows[end][2],
                'pause_changed_last5':any(rows[k][2]!=rows[k-1][2] for k in range(begin+1,end+1))}
        end=point(start+300)
        if end is not None:
            result['first5_growth']=rows[end][1]-rows[i][1]
            result['largest_minute_growth_first5']=max(rows[k][1]-rows[k-1][1] for k in range(i+1,end+1))
        details.append(result)
summary={}
for lead in (15,30,60):
    values=[d['before'][str(lead)] for d in details if d['before'][str(lead)] is not None]
    summary[str(lead)]={'available_events':len(values),
        'median_last5_growth':median(v['change_last5'] for v in values),
        'already_growing_5plus':sum(v['change_last5']>=5 for v in values),
        'pause_change_last5':sum(v['pause_changed_last5'] for v in values)}
report={'limitation':'descriptive event-only profile; not specificity, causal evidence or a validated forecast',
        'summary':summary,'quiet_controls':controls,'control_rule':'5min-spaced origins, no labeled event within +/-75min; overlapping dependent windows; not time-of-day matched',
        'events':details}
(root/'surge-precursors-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(summary))
print('quiet_controls',json.dumps(controls))
print('chop_case',json.dumps([d for d in details if d['checkpoint_id']==17 and d['event_start'].startswith('2026-10-01T17')]))


# surge-context-probe.py  Дослідне порівняння на вже погоджених даних. Повторно виконує базову пробу через runpy; мережевих запитів немає.

In [ ]:
"""Offline check of cross-checkpoint context using approved extracts only."""
import bisect,json,runpy,sys
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

root=Path('C:/AI/AvelRen/dist')
sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'))
rows=g['samples'](60,15)
groups=[[r for r in rows if r['origin']<g['VALIDATION']],
        [r for r in rows if g['VALIDATION']<=r['origin']<g['TEST']],
        [r for r in rows if r['origin']>=g['TEST']]]

def context(r):
    origin=r['origin'];values=[]
    for cp in g['CP']:
        t,c,p,bad,_,_=g['series'][cp]
        j=bisect.bisect_right(t,origin)-1
        if j<0 or origin-t[j]>90:return None
        deltas=[]
        for lag in (5,20):
            target=int(t[j])-lag*60
            k=bisect.bisect_left(t,target)
            candidates=[i for i in (k-1,k) if 0<=i<j]
            if not candidates:return None
            i=min(candidates,key=lambda i:abs(t[i]-target))
            if abs(t[i]-target)>30 or bad[i]!=bad[j]:return None
            deltas.append(c[j]-c[i])
        values.extend([c[j],*deltas,p[j]])
    return list(r['features'])+values

prepared=[]
for group in groups:
    good=[];features=[]
    for r in group:
        v=context(r)
        if v is not None:good.append(r);features.append(v)
    prepared.append((good,np.array(features)))
train,x=prepared[0];val,vx=prepared[1];test,tx=prepared[2]
y=np.array([r['y'] for r in train],dtype=float)
weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
report={'source_hashes':g['source_hashes'],'limitation':'exploratory reused test; not final holdout or calibrated probability',
        'context':'latest available counts, 5/20min changes and pause at all 4 checkpoints; no future values or gap filling',
        'split_samples':[len(train),len(val),len(test)],'methods':{}}
for name,columns in [('optimized_own_only',15),('cross_checkpoint_context',31)]:
    xx=x[:,:columns];mu=xx.mean(axis=0);sd=xx.std(axis=0);sd[sd<1e-8]=1
    def matrix(a):return np.column_stack((np.ones(len(a)),(a[:,:columns]-mu)/sd))
    design=matrix(x)
    def loss(coef):
        logits=design@coef;prob=1/(1+np.exp(-np.clip(logits,-30,30)))
        value=float(np.mean(weights*(np.logaddexp(0,logits)-y*logits))+.03/2*np.sum(coef[1:]**2))
        grad=design.T@((prob-y)*weights)/len(y);grad[1:]+=.03*coef[1:]
        return value,grad
    fit=minimize(loss,np.zeros(columns+1),jac=True,method='L-BFGS-B',options={'maxiter':200,'ftol':1e-9})
    def scores(a):return 1/(1+np.exp(-np.clip(matrix(a)@fit.x,-30,30)))
    vs=scores(vx);ts=scores(tx)
    thresholds=sorted(set([0.0,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,41)).tolist()]))
    choices=[(g['score_alerts'](val,vs,cut,15,60),cut) for cut in thresholds]
    selected,cut=max(choices,key=lambda c:(c[0]['f1'],c[0]['precision'],-c[0]['alerts']))
    result=g['score_alerts'](test,ts,cut,15,60)
    report['methods'][name]={'converged':bool(fit.success),'threshold':float(cut),'validation':selected,'test':result}
    print('CONTEXT',name,json.dumps(result),flush=True)
report['selected_by_validation']=max(report['methods'],key=lambda n:(report['methods'][n]['validation']['f1'],report['methods'][n]['validation']['precision'],-report['methods'][n]['validation']['alerts']))
(root/'surge-context-probe-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('CONTEXT_SELECTION',report['selected_by_validation'],flush=True)


# surge-own-probe.py  Дослідне порівняння на вже погоджених даних. Повторно виконує базову пробу через runpy; мережевих запитів немає.

In [ ]:
"""Offline own-checkpoint-only feature probe, no external registration schedule."""
import bisect,json,runpy,sys
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

root=Path('C:/AI/AvelRen/dist')
sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'))
rows=g['samples'](60,15)
splits=[[r for r in rows if r['origin']<g['VALIDATION']],
        [r for r in rows if g['VALIDATION']<=r['origin']<g['TEST']],
        [r for r in rows if r['origin']>=g['TEST']]]
state_starts={}
for cp,(times,counts,paused,*_) in g['series'].items():
    starts=np.zeros(len(times),dtype=np.int64)
    for j in range(1,len(times)):
        starts[j]=j if paused[j]!=paused[j-1] or times[j]-times[j-1]>90 else starts[j-1]
    state_starts[cp]=starts

def features(r,enriched):
    f=list(r['features'])
    if not enriched:return f
    local=datetime.fromtimestamp(r['origin'],timezone.utc).astimezone(g['zone'])
    day=[int(local.weekday()==k) for k in range(7)]
    timebins=[int(local.hour//6==k) for k in range(4)]
    times,counts,paused,*_=g['series'][r['cp']]
    j=bisect.bisect_right(times,r['origin'])-1
    age=min(180,(r['origin']-int(times[state_starts[r['cp']][j]]))/60)
    cp_flags=f[11:15]
    return f+day+timebins+[age,age*f[6],age*(1-f[6]),f[1]/max(1,f[0]),f[2]/max(1,f[0])]+[d*c for c in cp_flags for d in day]

report={'source_hashes':g['source_hashes'],'scope':'own checkpoint only; registration continuously available according to owner',
        'limitation':'exploratory reused test; no calibrated probability or 99% claim','methods':{}}
train,val,test=splits;y=np.array([r['y'] for r in train],dtype=float)
weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
for name,enriched in [('optimized_own_only',False),('own_weekday_state',True)]:
    arrays=[np.array([features(r,enriched) for r in rs],dtype=float) for rs in splits]
    mu=arrays[0].mean(axis=0);sd=arrays[0].std(axis=0);sd[sd<1e-8]=1
    design=[np.column_stack((np.ones(len(a)),(a-mu)/sd)) for a in arrays]
    x=design[0]
    def objective(coef):
        z=x@coef;p=1/(1+np.exp(-np.clip(z,-30,30)))
        loss=float(np.mean(weights*(np.logaddexp(0,z)-y*z))+.03/2*np.sum(coef[1:]**2))
        grad=x.T@((p-y)*weights)/len(y);grad[1:]+=.03*coef[1:]
        return loss,grad
    fit=minimize(objective,np.zeros(x.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':250,'ftol':1e-9})
    vs,ts=[1/(1+np.exp(-np.clip(a@fit.x,-30,30))) for a in design[1:]]
    cuts=sorted(set([0.0,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,41)).tolist()]))
    choices=[(g['score_alerts'](val,vs,cut,15,60),cut) for cut in cuts]
    selected,cut=max(choices,key=lambda c:(c[0]['f1'],c[0]['precision'],-c[0]['alerts']))
    result=g['score_alerts'](test,ts,cut,15,60)
    report['methods'][name]={'converged':bool(fit.success),'features':x.shape[1]-1,'threshold':float(cut),'validation':selected,'test':result}
    print('OWN',name,json.dumps(result),flush=True)
report['selected_by_validation']=max(report['methods'],key=lambda n:(report['methods'][n]['validation']['f1'],report['methods'][n]['validation']['precision'],-report['methods'][n]['validation']['alerts']))
(root/'surge-own-probe-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('OWN_SELECTION',report['selected_by_validation'],flush=True)


# surge-per-checkpoint-probe.py  Перевірка лише на погоджених локальних артефактах. Нульові історичні оцінки очікування невідомі. Нові мережеві запити не виконуються.

In [ ]:
"""Research classifier per checkpoint; current approved inputs only."""
import json,runpy,sys
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

root=Path('C:/AI/AvelRen/dist');sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'));rows=g['samples'](60,15)
train=[r for r in rows if r['origin']<g['VALIDATION']]
val=[r for r in rows if g['VALIDATION']<=r['origin']<g['TEST']]
test=[r for r in rows if r['origin']>=g['TEST']]
models={};vs=np.full(len(val),-1.);ts=np.full(len(test),-1.)
for cp in g['CP']:
    tr=[r for r in train if r['cp']==cp];x=np.array([r['features'][:11] for r in tr]);y=np.array([r['y'] for r in tr],dtype=float)
    if not y.sum() or y.sum()==len(y):
        models[str(cp)]={'status':'insufficient training classes','positive_samples':int(y.sum())};continue
    mu=x.mean(axis=0);sd=x.std(axis=0);sd[sd<1e-8]=1
    x=np.column_stack((np.ones(len(x)),(x-mu)/sd))
    weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
    def loss(coef):
        z=x@coef;p=1/(1+np.exp(-np.clip(z,-30,30)))
        value=float(np.mean(weights*(np.logaddexp(0,z)-y*z))+.03/2*np.sum(coef[1:]**2))
        grad=x.T@((p-y)*weights)/len(y);grad[1:]+=.03*coef[1:]
        return value,grad
    fit=minimize(loss,np.zeros(x.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':200,'ftol':1e-9})
    models[str(cp)]={'status':'fitted','converged':bool(fit.success),'positive_samples':int(y.sum())}
    for rs,output in ((val,vs),(test,ts)):
        ix=[i for i,r in enumerate(rs) if r['cp']==cp]
        xx=np.array([rs[i]['features'][:11] for i in ix]);xx=np.column_stack((np.ones(len(xx)),(xx-mu)/sd))
        output[ix]=1/(1+np.exp(-np.clip(xx@fit.x,-30,30)))
thresholds=sorted(set([0.,float(vs.max()+1e-9),*np.quantile(vs[vs>=0],np.linspace(0,1,41)).tolist()]))
choices=[(g['score_alerts'](val,vs,cut,15,60),cut) for cut in thresholds]
selected,cut=max(choices,key=lambda c:(c[0]['f1'],c[0]['precision'],-c[0]['alerts']))
result=g['score_alerts'](test,ts,cut,15,60)
report={'limitation':'exploratory reused test; missing model is unavailable, not evidence of no risk',
        'models':models,'threshold':float(cut),'validation':selected,'test':result}
(root/'surge-per-checkpoint-probe-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('PER_CHECKPOINT',json.dumps(report),flush=True)


# surge-own-probe.py  Перевірка лише на погоджених локальних артефактах. Нульові історичні оцінки очікування невідомі. Нові мережеві запити не виконуються.

In [ ]:
import sys
sys.argv = ['surge-own-probe.py', '--wait']
"""Offline own-checkpoint-only feature probe, no external registration schedule."""
import bisect,gzip,hashlib,json,runpy,sys
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

wait_enabled='--wait' in sys.argv
root=Path('C:/AI/AvelRen/dist')
sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'))
rows=g['samples'](60,15)
wait_values={}
if wait_enabled:
    wait_path=root/'queue-surge-wait-20260806-1001.json.gz'
    wait_rows=json.loads(gzip.decompress(wait_path.read_bytes()))
    wait_values={(cp,t):value for cp,t,value in wait_rows}
    assert len(wait_values)==len(wait_rows)==316136
    assert set(wait_values)=={(cp,int(t)) for cp,series in g['series'].items() for t in series[0]}
splits=[[r for r in rows if r['origin']<g['VALIDATION']],
        [r for r in rows if g['VALIDATION']<=r['origin']<g['TEST']],
        [r for r in rows if r['origin']>=g['TEST']]]
state_starts={}
for cp,(times,counts,paused,*_) in g['series'].items():
    starts=np.zeros(len(times),dtype=np.int64)
    for j in range(1,len(times)):
        starts[j]=j if paused[j]!=paused[j-1] or times[j]-times[j-1]>90 else starts[j-1]
    state_starts[cp]=starts

def features(r,enriched):
    f=list(r['features'])
    if not enriched:return f
    if enriched=='wait':
        cp,origin=r['cp'],r['origin'];times=g['series'][cp][0]
        current=wait_values[(cp,origin)]
        known=current>0
        additions=[current/3600 if known else np.nan,float(known)]
        for minutes in (5,20,60):
            target=origin-minutes*60;k=bisect.bisect_left(times,target)
            candidates=[i for i in (k-1,k) if 0<=i<len(times) and times[i]<=origin]
            i=min(candidates,key=lambda i:abs(times[i]-target))
            earlier=wait_values[(cp,int(times[i]))]
            valid=known and earlier>0 and abs(times[i]-target)<=30
            additions.extend([(current-earlier)/3600 if valid else np.nan,float(valid)])
        additions.extend([current/max(1,f[0])/3600 if known and f[0]>0 else np.nan,
                          float(known and f[0]>0)])
        return f+additions
    local=datetime.fromtimestamp(r['origin'],timezone.utc).astimezone(g['zone'])
    day=[int(local.weekday()==k) for k in range(7)]
    timebins=[int(local.hour//6==k) for k in range(4)]
    times,counts,paused,*_=g['series'][r['cp']]
    j=bisect.bisect_right(times,r['origin'])-1
    age=min(180,(r['origin']-int(times[state_starts[r['cp']][j]]))/60)
    cp_flags=f[11:15]
    return f+day+timebins+[age,age*f[6],age*(1-f[6]),f[1]/max(1,f[0]),f[2]/max(1,f[0])]+[d*c for c in cp_flags for d in day]

report={'source_hashes':g['source_hashes'],'scope':'own checkpoint only; registration continuously available according to owner',
        'limitation':'exploratory reused test; no calibrated probability or 99% claim','methods':{}}
train,val,test=splits;y=np.array([r['y'] for r in train],dtype=float)
weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
methods=[('optimized_own_only',False),('own_wait_estimate','wait') if wait_enabled else ('own_weekday_state',True)]
if wait_enabled:
    report['wait_source_sha256']=hashlib.sha256(wait_path.read_bytes()).hexdigest()
    report['missing_wait_rule']='historical zero is unknown; missing numerical features imputed by training medians, explicit validity flags retained'
for name,enriched in methods:
    arrays=[np.array([features(r,enriched) for r in rs],dtype=float) for rs in splits]
    if enriched=='wait':
        imputation=np.nanmedian(arrays[0],axis=0)
        imputation=np.where(np.isfinite(imputation),imputation,0)
        arrays=[np.where(np.isfinite(a),a,imputation) for a in arrays]
    mu=arrays[0].mean(axis=0);sd=arrays[0].std(axis=0);sd[sd<1e-8]=1
    design=[np.column_stack((np.ones(len(a)),(a-mu)/sd)) for a in arrays]
    x=design[0]
    def objective(coef):
        z=x@coef;p=1/(1+np.exp(-np.clip(z,-30,30)))
        loss=float(np.mean(weights*(np.logaddexp(0,z)-y*z))+.03/2*np.sum(coef[1:]**2))
        grad=x.T@((p-y)*weights)/len(y);grad[1:]+=.03*coef[1:]
        return loss,grad
    fit=minimize(objective,np.zeros(x.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':250,'ftol':1e-9})
    vs,ts=[1/(1+np.exp(-np.clip(a@fit.x,-30,30))) for a in design[1:]]
    cuts=sorted(set([0.0,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,41)).tolist()]))
    choices=[(g['score_alerts'](val,vs,cut,15,60),cut) for cut in cuts]
    selected,cut=max(choices,key=lambda c:(c[0]['f1'],c[0]['precision'],-c[0]['alerts']))
    result=g['score_alerts'](test,ts,cut,15,60)
    report['methods'][name]={'converged':bool(fit.success),'features':x.shape[1]-1,'threshold':float(cut),'validation':selected,'test':result}
    print('OWN',name,json.dumps(result),flush=True)
report['selected_by_validation']=max(report['methods'],key=lambda n:(report['methods'][n]['validation']['f1'],report['methods'][n]['validation']['precision'],-report['methods'][n]['validation']['alerts']))
result_path=root/('surge-wait-probe-results.json' if wait_enabled else 'surge-own-probe-results.json')
result_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('OWN_SELECTION',report['selected_by_validation'],flush=True)


## Історичні аналоги: одночасна ціль 99% / 99%
Вимога власника: 99% справджених попереджень і 99% завчасно передбачених подій. Жоден кандидат її не досяг. Добір за validation, тест дослідний і повторно використаний. Горизонт 15–60 хв не замінює наступну добу.


In [ ]:
"""Local research only: historical analogs, train-only reference library.

No calibrated probability; overlapping reference rows are not independent
events. Reused exploratory test cannot establish final 99% acceptance.
"""
import json, runpy, sys
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

root=Path('C:/AI/AvelRen/dist')
sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'))
rows=g['samples'](60,15)
train=[r for r in rows if r['origin']<g['VALIDATION']]
val=[r for r in rows if g['VALIDATION']<=r['origin']<g['TEST']]
test=[r for r in rows if r['origin']>=g['TEST']]
report={'source_hashes':g['source_hashes'],
        'target':'precision >=0.99 AND event recall >=0.99',
        'horizon':'15-60 minutes, not full rolling 24h forecast',
        'limitation':'reused exploratory test; correlated minute reference rows; scores uncalibrated',
        'methods':{}}
for neighbors in (20,100,500):
    scores=[np.zeros(len(val)),np.zeros(len(test))]
    for cp in g['CP']:
        reference=[r for r in train if r['cp']==cp]
        x=np.array([r['features'][:11] for r in reference])
        labels=np.array([r['y'] for r in reference])
        mu=x.mean(axis=0);sd=x.std(axis=0);sd[sd<1e-8]=1
        tree=cKDTree((x-mu)/sd)
        for target,output in zip((val,test),scores):
            ids=[i for i,r in enumerate(target) if r['cp']==cp]
            q=np.array([target[i]['features'][:11] for i in ids])
            distances,indices=tree.query((q-mu)/sd,k=min(neighbors,len(reference)),workers=1)
            weights=1/(.1+distances)
            output[ids]=(weights*labels[indices]).sum(axis=1)/weights.sum(axis=1)
    vs,ts=scores
    # Fixed threshold grid: 101 quantiles chosen on validation only.
    cuts=sorted(set([0.,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,101)).tolist()]))
    choices=[(g['score_alerts'](val,vs,c,15,60),c) for c in cuts]
    chosen,cut=max(choices,key=lambda z:(z[0]['f1'],z[0]['precision'],-z[0]['alerts']))
    result=g['score_alerts'](test,ts,cut,15,60)
    both=[(m,c) for m,c in choices if m['precision']>=.99 and m['recall']>=.99]
    name=f'own_analogs_{neighbors}'
    report['methods'][name]={'threshold':cut,'validation':chosen,'test':result,
                            'joint_99_validation_threshold_available':bool(both)}
    print('ANALOG',name,json.dumps(result),flush=True)
report['selected_by_validation']=max(report['methods'],key=lambda n:(report['methods'][n]['validation']['f1'],report['methods'][n]['validation']['precision'],-report['methods'][n]['validation']['alerts']))
(root/'surge-analog-probe-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('SELECTED',report['selected_by_validation'],flush=True)


## Історія 2–4 години та рухомий добовий горизонт
Добовий тест: 2/3 придатних подій, 1 хибне попередження. Два напливи Чопа 01.10 виключені з добової оцінки. Це класифікація широкого вікна, не точний час і не незалежний тест. 99% / 99% не досягнуто.


In [ ]:
"""Offline research: long past history, short and rolling 24h event labels.

Purged chronological splits; train-only scaling. Test already reused.
24h score means at least one event in 1-24h, not predicted event time.
"""
import bisect,json,runpy,sys
from pathlib import Path
import numpy as np
from scipy.optimize import minimize

root=Path('C:/AI/AvelRen/dist')
sys.argv=['surge-model-probe.py','--older']
g=runpy.run_path(str(root/'surge-model-probe.py'))
base=g['samples'](60,15)
report={'source_hashes':g['source_hashes'],'target':'precision >=99% AND recall >=99%',
        'limitations':['reused exploratory test','net growth, not gross registrations',
                       '24h classification does not provide event time or magnitude'],
        'experiments':{}}

def long_features(r):
    t,c,p,bad,severe,events=g['series'][r['cp']]
    j=bisect.bisect_left(t,r['origin']);out=[]
    for minutes in (120,240):
        goal=r['origin']-minutes*60;k=bisect.bisect_left(t,goal)
        options=[v for v in (k-1,k) if 0<=v<=j]
        if not options:return None
        i=min(options,key=lambda v:abs(t[v]-goal))
        if abs(t[i]-goal)>30 or severe[i]!=severe[j] or (j-i+1)/(minutes+1)<.95:return None
        values=c[i:j+1];changes=np.diff(values)
        out.extend([values[-1]-values[0],values.std(),values.max()-values.min(),
                    np.maximum(changes,0).sum(),np.maximum(-changes,0).sum(),
                    (changes>0).mean(),np.mean(p[i:j+1]),np.sum(np.diff(p[i:j+1])!=0)])
    return list(r['features'])+out

for horizon,lead in ((60,15),(1440,60)):
    origins=[]
    for original in base:
        r=dict(original)
        if horizon==1440:
            local=g['datetime'].fromtimestamp(r['origin'],g['timezone'].utc).astimezone(g['zone'])
            if local.minute!=0:continue
            t,c,p,bad,severe,events=g['series'][r['cp']]
            j=bisect.bisect_left(t,r['origin']);end=r['origin']+horizon*60+300
            boundary=g['VALIDATION'] if r['origin']<g['VALIDATION'] else g['TEST'] if r['origin']<g['TEST'] else g['END']
            if end>=boundary:continue
            k=bisect.bisect_left(t,end)
            if k>=len(t) or abs(t[k]-end)>30 or severe[k]!=severe[j] or (k-j+1)/((end-r['origin'])/60+1)<.95:continue
            r['events']=events[(events>=r['origin']+lead*60)&(events<=r['origin']+horizon*60)].tolist()
            r['y']=int(bool(r['events']))
        origins.append(r)
    enriched=[(r,long_features(r)) for r in origins]
    common=[r|{'long_features':f} for r,f in enriched if f is not None]
    splits=[[r for r in common if r['origin']<g['VALIDATION']],
            [r for r in common if g['VALIDATION']<=r['origin']<g['TEST']],
            [r for r in common if r['origin']>=g['TEST']]]
    train,val,test=splits
    raw_test_events={(cp,int(e)) for cp,series in g['series'].items() for e in series[-1] if g['TEST']<=e<g['END']}
    eligible_test_events={(r['cp'],e) for r in test for e in r['events']}
    experiment={'origin_rows_before_long_history':len(origins),'common_rows':len(common),
                'split_rows':[len(s) for s in splits],
                'positive_rows':[sum(r['y'] for r in s) for s in splits],
                'test_events_before_long_history':len({(r['cp'],e) for r in origins if r['origin']>=g['TEST'] for e in r['events']}),
                'test_events_after_long_history':len({(r['cp'],e) for r in test for e in r['events']}),
                'all_labeled_events_in_test':len(raw_test_events),
                'events_excluded_from_test_recall':[{'checkpoint_id':cp,'event_start':g['datetime'].fromtimestamp(e,g['timezone'].utc).astimezone(g['zone']).isoformat()} for cp,e in sorted(raw_test_events-eligible_test_events)],
                'methods':{}}
    assert all(not r['events'] or min(r['events'])>=r['origin']+lead*60 for r in common)
    if not all(splits) or not 0<sum(r['y'] for r in train)<len(train):
        experiment['status']='insufficient_labels';report['experiments'][str(horizon)]=experiment;continue
    y=np.array([r['y'] for r in train],dtype=float)
    weights=np.where(y==1,len(y)/(2*y.sum()),len(y)/(2*(len(y)-y.sum())))
    for name,key in (('baseline_common_origins','features'),('past_2_4_hours','long_features')):
        arrays=[np.array([r[key] for r in s]) for s in splits]
        mu=arrays[0].mean(axis=0);sd=arrays[0].std(axis=0);sd[sd<1e-8]=1
        design=[np.column_stack((np.ones(len(a)),(a-mu)/sd)) for a in arrays]
        x=design[0]
        def objective(coef):
            z=x@coef;p=1/(1+np.exp(-np.clip(z,-30,30)))
            loss=np.mean(weights*(np.logaddexp(0,z)-y*z))+.03/2*np.sum(coef[1:]**2)
            grad=x.T@((p-y)*weights)/len(y);grad[1:]+=.03*coef[1:]
            return float(loss),grad
        fit=minimize(objective,np.zeros(x.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':250,'ftol':1e-9})
        assert fit.success,fit.message
        vs,ts=[1/(1+np.exp(-np.clip(a@fit.x,-30,30))) for a in design[1:]]
        cuts=sorted(set([0.,float(vs.max()+1e-9),*np.quantile(vs,np.linspace(0,1,101)).tolist()]))
        choices=[(g['score_alerts'](val,vs,c,lead,horizon),c) for c in cuts]
        selected,cut=max(choices,key=lambda z:(z[0]['f1'],z[0]['precision'],-z[0]['alerts']))
        result=g['score_alerts'](test,ts,cut,lead,horizon)
        experiment['methods'][name]={'threshold':cut,'converged':bool(fit.success),'validation':selected,'test':result,
                                    'joint_99_validation':any(m['precision']>=.99 and m['recall']>=.99 for m,c in choices)}
        print('LONG',horizon,name,json.dumps(result),flush=True)
    experiment['selected_by_validation']=max(experiment['methods'],key=lambda n:(experiment['methods'][n]['validation']['f1'],experiment['methods'][n]['validation']['precision'],-experiment['methods'][n]['validation']['alerts']))
    report['experiments'][str(horizon)]=experiment
(root/'surge-long-history-probe-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('LONG_SAVED',flush=True)


## Зафіксована модель: незалежний період 02–09.10
Модель зафіксована до перегляду нових результатів. Перші 48 моментів очікують завершення майбутнього вікна; точність поки невідома. Параметри наведено для збереження відтворюваності. Не продуктове розгортання.
```json
{
  "kind": "research_only_frozen_baseline_24h",
  "source_hashes": {
    "queue-surge-28days.json.gz": "3ca0e0f32e967278e2710afdb993ca0b36c073149fb5121d9d16e7b7983a9a2f",
    "queue-surge-older-20260806-0903.json.gz": "19a0a1d59a94de4918ad4600c5b40daa856ba287adce32e8666f0702ad40db62"
  },
  "probe_source_sha256": "9afe0a21467e333b64c7022adc9b0e0944c978f98cfb56a2f02654ddb3da6b1f",
  "trained_before": "2026-09-18T00:00:00+03:00",
  "threshold_selected_before": "2026-09-25T00:00:00+03:00",
  "holdout_origins_start": "2026-10-02T00:00:00+03:00",
  "holdout_origins_end_exclusive": "2026-10-09T00:00:00+03:00",
  "required_future_data_until": "2026-10-10T00:05:00+03:00",
  "feature_mean": [
    823.440366972477,
    0.01452599388379205,
    -0.009938837920489297,
    -0.21788990825688073,
    -1.0603975535168195,
    0.0194954128440367,
    0.573394495412844,
    0.05259539474205046,
    0.04703383734206769,
    -0.06412982549058296,
    0.025705036610285757,
    0.25,
    0.25,
    0.25,
    0.25
  ],
  "feature_std": [
    463.3049737888426,
    3.193519336705452,
    5.86128324521708,
    9.184479553064639,
    22.779410679652976,
    1.431809569905992,
    0.49458391405614005,
    0.7176265583528888,
    0.6928446184704034,
    0.7156707255359495,
    0.695012251105166,
    0.4330127018922193,
    0.4330127018922193,
    0.4330127018922193,
    0.4330127018922193
  ],
  "coefficients": [
    -1.224828365408305,
    -1.3978514012393164,
    -0.07813907882897021,
    -0.11239749610154594,
    0.12261680638152672,
    -0.0019422593329956861,
    0.05577426135288847,
    -0.11452411072182617,
    -0.09762832486588631,
    0.02863706186626831,
    -0.1909153343367414,
    -0.6997784523237947,
    -0.4795427763865135,
    0.1627476056936668,
    -0.14330040867831578,
    0.4600955793711633
  ],
  "threshold": 0.8789563370805348,
  "feature_order": "count,d5,d10,d20,d60,d5-d10/2,paused,sin_day,cos_day,sin_week,cos_week,cp1,cp6,cp17,cp29",
  "origin_frequency_minutes": 60,
  "lead_minutes": 60,
  "horizon_minutes": 1440,
  "past_quality": "baseline last60min no gap>90s; additionally last120/240min coverage>=95%, no gap>300s, endpoint tolerance30s",
  "future_quality": "24h+5min coverage>=95%, no gap>300s, endpoint tolerance30s",
  "event_definition": "net +20 within5min, no internal gap>90s, endpoint tolerance30s; windows within30min clustered",
  "matching": "new alert cooldown60min per checkpoint; chronological one alert to one unmatched event",
  "acceptance": "precision>=0.99 AND recall>=0.99; excluded events reported separately; sufficient independent evidence required",
  "scope_limitation": "predicts any event in1-24h, not precise event time or magnitude; not deployed"
}
```


In [ ]:
import sys
sys.argv=['surge-frozen-evaluate.py','--new-data','C:/AI/AvelRen/dist/queue-surge-holdout-20261002-1010.json.gz']
"""Research-only frozen inference and later evaluation; no fitting/network.

Forecast issuance uses past only. Unknown future is pending/unassessable,
never silently a negative. Wide-window classifier is not precise timing.
"""
import argparse,bisect,gzip,hashlib,json,math
from collections import defaultdict
from datetime import datetime,timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np

root=Path('C:/AI/AvelRen/dist');zone=ZoneInfo('Europe/Kyiv')
ap=argparse.ArgumentParser()
ap.add_argument('--new-data',type=Path)
ap.add_argument('--start',default='2026-10-02T00:00:00+03:00')
ap.add_argument('--end',default='2026-10-09T00:00:00+03:00')
ap.add_argument('--output',type=Path,default=root/'surge-frozen-evaluation.json')
args=ap.parse_args()
model_path=root/'surge-frozen-24h-20261002.json'
model=json.loads(model_path.read_text());model_hash=hashlib.sha256(model_path.read_bytes()).hexdigest()
assert model_hash=='331320e05c62ec027744dab387ba40c859a9c1070821afc3910f4358bb3b90bb','frozen model changed'
start=int(datetime.fromisoformat(args.start).timestamp());end=int(datetime.fromisoformat(args.end).timestamp())
assert start<end
mu=np.array(model['feature_mean']);sd=np.array(model['feature_std']);coef=np.array(model['coefficients'])
assert len(mu)==len(sd)==15 and len(coef)==16 and np.all(sd>0)
data=defaultdict(list);keys=set();sources={}
paths=[root/name for name in model['source_hashes']]
if args.new_data:paths.append(args.new_data)
for path in paths:
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if path.name in model['source_hashes']:assert digest==model['source_hashes'][path.name]
    sources[path.name]=digest
    for cp,t,c,p in json.loads(gzip.decompress(path.read_bytes())):
        assert cp in (1,6,17,29) and c>=0 and isinstance(p,bool)
        assert (cp,t) not in keys,'duplicate input key'
        keys.add((cp,t));data[cp].append((t,c,p))

predictions=[];eligible=set();events_all=set();quality_rejections=0;pending_origins=0
for cp,rows in data.items():
    rows.sort();times=np.array([r[0] for r in rows]);counts=np.array([r[1] for r in rows]);paused=np.array([r[2] for r in rows])
    def closest(goal):
        k=bisect.bisect_left(times,goal);options=[v for v in (k-1,k) if 0<=v<len(times)]
        return min(options,key=lambda v:abs(times[v]-goal))
    def good(i,j,minutes,max_gap,coverage=True):
        return j>=i and abs(times[j]-times[i]-minutes*60)<=60 and (not coverage or (j-i+1)/(minutes+1)>=.95) and np.all(np.diff(times[i:j+1])<=max_gap)
    events=[];last_window=-10**20
    for j,t in enumerate(times):
        i=closest(t-300)
        if i<j and abs(times[i]-(t-300))<=30 and good(i,j,5,90,False) and counts[j]-counts[i]>=20:
            s=int(times[i])
            if s>last_window+1800:events.append(s)
            last_window=s
    events_all.update((cp,e) for e in events if start<=e<end)
    for j,t in enumerate(times):
        if not start<=t<end:continue
        local=datetime.fromtimestamp(int(t),timezone.utc).astimezone(zone)
        if local.minute!=0:continue
        lags=[]
        for minutes in (5,10,20,60):
            i=closest(t-minutes*60)
            if i>j or abs(times[i]-(t-minutes*60))>30 or not good(i,j,minutes,90,False):break
            lags.append(float(counts[j]-counts[i]))
        if len(lags)!=4:quality_rejections+=1;continue
        long_ok=True
        for minutes in (120,240):
            i=closest(t-minutes*60)
            if abs(times[i]-(t-minutes*60))>30 or not good(i,j,minutes,300):long_ok=False;break
        if not long_ok:quality_rejections+=1;continue
        phase=local.hour*60+local.minute
        f=np.array([counts[j],*lags,lags[0]-lags[1]/2,float(paused[j]),
                    math.sin(2*math.pi*phase/1440),math.cos(2*math.pi*phase/1440),
                    math.sin(2*math.pi*local.weekday()/7),math.cos(2*math.pi*local.weekday()/7),
                    *(int(cp==v) for v in (1,6,17,29))])
        z=coef[0]+((f-mu)/sd)@coef[1:];score=float(1/(1+np.exp(-np.clip(z,-30,30))))
        label_end=int(t)+1440*60+300
        if label_end>times[-1]:status='pending_future';pending_origins+=1
        else:
            k=closest(label_end)
            status='assessable' if abs(times[k]-label_end)<=30 and good(j,k,1445,300) else 'future_data_gap'
        reachable=[e for e in events if t+3600<=e<=t+86400] if status=='assessable' else []
        eligible.update((cp,e) for e in reachable)
        predictions.append({'cp':cp,'origin':int(t),'score':score,'future_status':status,'events':reachable})

alerts=[];last={}
for p in sorted(predictions,key=lambda v:(v['origin'],v['cp'])):
    if p['score']>=model['threshold'] and p['origin']-last.get(p['cp'],-10**20)>=3600:
        alerts.append(dict(p));last[p['cp']]=p['origin']
matched=set();correct=0;false=0
for p in alerts:
    if p['future_status']!='assessable':p['outcome']='not_yet_assessable';continue
    possibilities=[e for e in p['events'] if (p['cp'],e) not in matched]
    if possibilities:
        e=min(possibilities);matched.add((p['cp'],e));correct+=1;p['outcome']='matched';p['matched_event']=e
    else:false+=1;p['outcome']='unmatched'
out={'model_sha256':model_hash,'source_hashes':sources,'origins_interval':[args.start,args.end],
     'status':'research_only; no 99% acceptance; event time/magnitude not predicted',
     'origin_scores':predictions,
     'predictions':len(predictions),'pending_origins':pending_origins,'past_quality_rejections':quality_rejections,
     'alerts':alerts,'correct_alerts':correct,'false_alerts':false,
     'unassessable_alerts':len(alerts)-correct-false,'eligible_events':len(eligible),'detected_events':len(matched),
     'observed_labeled_events_in_origin_interval':len(events_all),
     'observed_events_without_assessable_origin':[{'cp':cp,'time':e} for cp,e in sorted(events_all-eligible)],
     'precision_on_assessable_alerts':correct/(correct+false) if correct+false else None,
     'recall_on_eligible_events':len(matched)/len(eligible) if eligible else None,
     'note':'Unknown future and data gaps are not counted as negatives; cannot claim overall recall from eligible-only denominator.'}
args.output.write_text(json.dumps(out,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in out.items() if k not in ('alerts','origin_scores')},indent=2))
